# Trajectories and Spatial Gene Modules

`01_core_workflow.ipynb` ended with nine Leiden clusters. Leiden always returns discrete bins, even when the underlying biology is a continuum. This notebook examines whether the clusters represent genuinely distinct populations or cuts through a gradient: PAGA tests cluster connectivity in expression space; diffusion pseudotime (DPT) orders cells along any continuum PAGA reveals.

The second half addresses a separate question: genes that co-vary spatially (rising and falling together across the tissue) indicate shared regulatory programs or shared cell-type residency. `tl.spatial_gene_modules` finds these groups directly from the spatial graph, without cluster labels or pseudotime.

Dataset: `ab.datasets.xenium()`, re-processed here so this notebook runs standalone.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

## 1. Setup

Cluster identities from `01_core_workflow.ipynb` are the input to both trajectory analysis and gene-module discovery below, so this notebook reproduces that setup (load, filter, normalize, cluster) standalone rather than depending on the earlier notebook having run. `tl.cluster` and `tl.umap` both default to `neighbors_key='neighbors'` (the transcriptomic graph), passed explicitly below anyway for clarity -- pass `neighbors_key='spatial_neighbors'` instead for spatial-domain clustering.

In [ ]:
mdata = ab.datasets.xenium()
ab.pipeline.filter_xenium(mdata)

ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata, n_top_genes=2000)

ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)  # transcriptomic graph -> uns['neighbors']
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors')

mdata['xenium'].obs['leiden'].value_counts()

## 2. Testing cluster discreteness with PAGA

`tl.paga` abstracts the transcriptomic neighbor graph into a cluster-level connectivity graph: for every pair of Leiden clusters, normalized inter-cluster edge density is computed. Strong connectivity between two clusters indicates a continuum; weak connectivity everywhere indicates genuinely distinct populations.

`neighbors_key='neighbors'` is passed because PAGA must abstract the *expression* graph, not the spatial one.

In [ ]:
ab.tl.paga(mdata, groupby='leiden', neighbors_key='neighbors')
ab.pl.paga_plot(mdata, threshold=0.05).show()

The PAGA graph at threshold=0.05 reveals which clusters share intermediate cell states: thick edges mean high normalized connectivity between two clusters, a hallmark of a continuum rather than two discrete populations. Clusters that turn out to be biologically related (the same broad lineage, e.g. several epithelial subtypes, or several immune subtypes) are the ones worth expecting strong connectivity between -- but check `ab.tl.rank_genes_groups(mdata, 'leiden')` (as in `01_core_workflow.ipynb`) for what each cluster number actually is before predicting or explaining a connectivity pattern from its number alone; cluster identity is not fixed across independent clustering runs, including runs of this exact notebook with different HVG/PCA settings than others in this series.

Thick edges indicate clusters with high normalized connectivity — a hallmark of a continuum rather than two discrete populations. Clusters connected only by thin edges (below `threshold=0.05`) appear as expression-space islands with no intermediate cell states.

A hub-and-spoke topology is common in complex tissues: a broadly plastic state connected to several committed populations. Truly disconnected clusters indicate sharp fate boundaries rather than gradients. If all clusters are disconnected at this threshold, diffusion pseudotime has no meaningful axis to project onto.

## 3. Ordering cells along the trajectory

Diffusion pseudotime requires a root cell: DPT measures distance *from* that cell, so "early" and "late" are undefined without one. The root here is cluster 0, set via `root_obs_key='leiden'`/`root_obs_val='0'`; the first cell in that group is used automatically -- an arbitrary but reproducible choice, not a claim that cluster 0 is a biological point of origin (an FFPE adult tissue section has no developmental root to anchor to in the first place). `tl.dpt` runs `sc.tl.diffmap` internally if it hasn't been computed.

In [ ]:
ab.tl.dpt(mdata, root_obs_key='leiden', root_obs_val='0', neighbors_key='neighbors')

mdata['xenium'].obs['dpt_pseudotime'].describe()

Read skew from the `describe()` output directly: if the mean sits below the median, the distribution is left-skewed with its long tail toward low values; if the mean sits above the median, it's right-skewed with the long tail toward high values. The gap between the median and each quartile tells you which side that tail is on -- a wide median-to-25th-percentile gap versus a narrow median-to-75th-percentile gap (or vice versa) is the same signal in different units. Since DPT pseudotime is distance from the root cell, values near 0 mark cells transcriptionally close to the root; the shape of this distribution describes how compact vs. dispersed the cell population is at each end of the trajectory, not which biological compartment is 'larger.' If pseudotime aligns with the tissue's spatial axis, the overlay below should show a smooth color gradient rather than sharp discontinuities at cluster boundaries.

Pseudotime is stored in `obs['dpt_pseudotime']` and can be passed directly to `pl.points(color=...)`. Comparing it against discrete Leiden labels on the tissue tests the continuum hypothesis: pseudotime that varies smoothly across cluster boundaries rather than jumping at the same lines Leiden drew is evidence that the gradient captures biology the hard labels discard.

The first `pl.points()` call below returns a lazy `SpatialPanel` — nothing renders until `.show()` is called.

In [ ]:
# Not rendered yet -- pseudotime_panel is just a lazy SpatialPanel until .show()/repr triggers a draw.
pseudotime_panel = ab.pl.points(mdata, color='dpt_pseudotime')
type(pseudotime_panel)

In [ ]:
(pseudotime_panel | ab.pl.points(mdata, color='leiden')).show()

Comparing pseudotime to Leiden shows how well the two representations agree spatially. Where pseudotime varies smoothly *across* a Leiden cluster boundary (the color gradient doesn't jump at the same line as the cluster label), the discrete cluster label is cutting through a continuum rather than capturing a sharp biological transition. Where pseudotime and Leiden boundaries align closely (same spatial edge), the discrete label genuinely captures a cell-state boundary. `pl.violin(mdata, keys='dpt_pseudotime', groupby='leiden')` is a more direct way to check this than eyeballing two spatial maps side by side -- it shows whether pseudotime actually varies monotonically-ish across clusters or whether the claimed continuum doesn't hold up numerically.

## 4. Co-varying spatial gene programs

Trajectories describe continuity in *cell* identity. A separate question applies to *genes*: whether any are spatially co-expressed, rising and falling together across tissue, independently of cluster or pseudotime assignment. `tl.spatial_gene_modules` finds such groups from the spatial graph alone, without trajectory or cluster labels.

The function reads `obsp['{neighbors_key}_connectivities']` from a spatial graph, not the transcriptomic graph used for PAGA/DPT. `pipeline.filter_xenium` already built one during filtering (under `obsp['spatial_neighbors_connectivities']`), which matches the default `neighbors_key='spatial_neighbors'`.

In [ ]:
modules = ab.tl.spatial_gene_modules(mdata, n_modules=8)
modules.groupby('module').size()

The module sizes above are lopsided by design of this kind of clustering, not necessarily by biology: a hierarchical clustering into a fixed number of groups (`n_modules=8` here) always returns exactly 8 groups whether or not 8 real co-expression programs exist, and genes with little to no real signal in this tissue have nowhere else to go but into one large, low-signal catch-all. A module many times larger than the others, spanning genes from wildly different, tissue-implausible cell types (e.g. kidney, pancreatic islet, and erythroid markers all showing up together in a lung tissue module), is the signature of exactly that catch-all -- not a real "housekeeping" program. Check a module's actual gene list and its spatial score (plotted below) before trusting its size or interpreting it biologically; a tightly co-expressed real program is usually much smaller.

Each module's per-cell score is in `obsm['spatial_module_scores']` — a `(n_cells, n_modules)` array, one column per module, computed as the mean expression of that module's genes. A spatially coherent pattern in the highest-scoring cells (rather than diffuse noise) confirms the module reflects genuine co-expressed structure.

In [ ]:
largest_module = modules.groupby('module').size().idxmax()
smallest_module = modules.groupby('module').size().idxmin()
print(f"largest module: {largest_module} ({(modules['module'] == largest_module).sum()} genes)")
print(f"smallest module: {smallest_module} ({(modules['module'] == smallest_module).sum()} genes)")

mdata['xenium'].obs['largest_module_score'] = mdata['xenium'].obsm['spatial_module_scores'][:, largest_module]
mdata['xenium'].obs['smallest_module_score'] = mdata['xenium'].obsm['spatial_module_scores'][:, smallest_module]

# Contrast the largest module against the smallest -- the smaller one is the
# better candidate for a real, tightly co-expressed spatial program (see markdown below).
(
    ab.pl.points(mdata, color='largest_module_score', title=f'module {largest_module} (largest)')
    | ab.pl.points(mdata, color='smallest_module_score', title=f'module {smallest_module} (smallest)')
).show()

The two panels above are a direct contrast. The smaller module should show a spatially coherent pattern -- a compact, contiguous region of high score, matching a real anatomical structure -- if it reflects genuine co-expressed biology. The larger module frequently looks diffuse and near-uniform instead, consistent with it being the noise/off-tissue catch-all described above rather than a real spatial program. If the larger module shows a coherent pattern instead of diffuse noise on your own re-run, that's evidence its genes are genuinely broadly co-expressed rather than off-tissue -- check the gene list either way rather than assuming biological meaning from module size alone.

## 5. Self-organizing maps

Self-organizing maps (SOMs) are a topology-preserving dimensionality reduction technique that maps high-dimensional data onto a fixed 2-D grid of nodes. Unlike UMAP or PCA, which produce a continuous embedding, the SOM grid is discrete and reproducible, making it useful for three distinct analyses:

1. **Cell clustering** — each cell is assigned to its best-matching node (BMU). Nearby nodes hold similar cells; the U-matrix (unified distance matrix) reveals natural cluster boundaries without external labels.
2. **Gene module discovery** — train on the transposed matrix (genes as observations, cells as features) to group co-regulated genes without requiring a spatial graph.
3. **Subcellular spatial domains** — train on normalized intracellular transcript positions to find recurring spatial patterns *within* cells (nuclear accumulation, peripheral ring, polar asymmetry, etc.).

Requires the `minisom` optional dependency (`uv add 'astrolabe[som]'`).

### 5a. Cell-level SOM clustering

`tl.som` auto-sizes the grid using the heuristic `ceil(sqrt(5 × sqrt(n_cells)))`: for 154,270 cells this gives a 45×45 grid (2,025 nodes) and defaults to 100 × n_cells = 15.4 million training iterations, both impractical for a tutorial run. The example below uses `x=20, y=20` (400 nodes, enough to resolve the 9 Leiden clusters and their sub-states) and caps `n_iter=100_000` (about 0.65 random passes through the data, sufficient for a converged cluster topology on pre-reduced PCA). Setting `n_modules=16` in `tl.som_gene_modules` provides finer gene grouping than the 8 used for spatial modules because gene-level SOM operates on 377 observations (genes) rather than 154k cells.

In [ ]:
# Train a SOM on the PCA embedding.
# x=20, y=20: 400 nodes (enough to resolve 9+ clusters and sub-states).
# n_iter=100_000: caps training at a manageable wall time; auto-default of
# 100*n_cells = 15.4M iterations would take hours on 154k cells.
ab.tl.som(mdata, use_rep='X_pca', x=20, y=20, n_iter=100_000, seed=42)

xen = mdata['xenium']
print(f"Grid: {xen.uns['som']['params']['x']}x{xen.uns['som']['params']['y']}")
xen.obs['som_cluster'].value_counts().sort_index()

The 20×20 grid assigns 154,270 cells to 400 nodes — a mean of ~386 cells per node, ranging from ~151 to ~683. Node population spread indicates transcriptional density: densely populated nodes hold common cell states (likely the large tumor epithelial clusters 0 and 1), while sparsely populated nodes correspond to rare or transitional states. With 400 nodes against 9 Leiden clusters, the SOM resolves sub-cluster heterogeneity that Leiden's single-resolution partition misses — adjacent nodes that Leiden lumped into one cluster may represent sub-populations with distinct marker profiles. The U-matrix and hit-map below show where these boundaries and density peaks lie.

In [ ]:
# U-matrix reveals cluster boundaries (high = boundary between distinct cell states)
# Hit map shows how many cells land on each node (dense = common state)
(ab.pl.som_umatrix(mdata) | ab.pl.som_hits(mdata)).show()

The U-matrix encodes the mean distance from each node to its immediate grid neighbors: bright ridges mark transcriptional boundaries between distinct cell states; dark valleys mark plateaus of similar cells. A hit map that is unevenly concentrated (a few very dense nodes surrounded by sparse ones) confirms the dataset is dominated by a small number of abundant cell types. Comparing the U-matrix ridge pattern against the spatial tissue view (next cell) reveals whether transcriptional boundaries correspond to spatial tissue compartment edges: aligned boundaries indicate anatomy-driven cell identity, while scattered boundaries suggest continuous transcriptional variation independent of tissue structure.

In [ ]:
# SOM clusters in tissue space — compare with expression-based Leiden clusters
ab.pl.points(mdata, color='som_cluster').show()

The SOM tissue map uses 400 distinct colors (one per grid node), so cells assigned to nearby nodes in expression space appear in similar hues — a smooth color gradient in the tissue plot indicates a transcriptional continuum, while sharp color jumps at anatomical boundaries (tumor nest edges, stromal–immune interfaces) confirm those are real cell-state transitions. Comparing this to the 9-color Leiden map: regions that are uniform in Leiden but show SOM color variation contain the intra-cluster heterogeneity the discrete partition suppressed. If the SOM and Leiden boundaries coincide everywhere, the resolution chosen for SOM (400 nodes) is not revealing additional sub-structure beyond what Leiden captured.

### 5b. Gene modules via SOM

`tl.som_gene_modules` trains on the gene-by-cell matrix (each gene as an observation, cells as features) to cluster genes into co-expression modules. Unlike `tl.spatial_gene_modules` it requires no spatial graph — it captures purely transcriptomic co-regulation.


In [ ]:
som_genes = ab.tl.som_gene_modules(mdata, n_modules=16, seed=42)
print(som_genes.groupby('module').size().sort_values(ascending=False))

# Inspect the most-populated module
largest_mod = som_genes.groupby('module').size().idxmax()
print(f'Largest module ({largest_mod}):', som_genes.loc[som_genes['module'] == largest_mod, 'gene'].tolist()[:10])

Module `2_0` absorbs 231 of 377 genes (61%): a dominance pattern expected for a targeted Xenium panel where many genes were selected precisely because they co-vary with the same abundant cell types (epithelial markers all rise together in tumor nests). The two 21-gene modules (`1_0`, `2_1`) and the 13-gene module (`3_0`) are the most gene-specific; these smaller modules are likely better single-module discriminators for immune or stromal sub-populations than the giant bulk module. The first genes listed in module `2_0` (ABCC11, ACKR1, ADGRE1, ADH1C, ADIPOQ, AGER, AHSP, ALAS2, ALDH1A3, spanning macrophage receptors, hemoglobin biosynthesis, and adipokines) reflect diverse biology co-loaded into a catch-all module rather than a coherent program; the smaller modules deserve inspection for cell-type-specific regulons.

In [ ]:
# Score each cell for each gene module and show the top-scoring module spatially
scores = xen.obsm['som_gene_modules_scores']
top_mod_idx = scores.mean(axis=0).argmax()
xen.obs['top_som_module_score'] = scores[:, top_mod_idx]
ab.pl.points(mdata, color='top_som_module_score').show()

The top-scoring SOM gene module's spatial pattern tests whether the module captures a coherent tissue program. Because module `2_0` contains 61% of genes including many broadly expressed markers, its score is expected to be high everywhere — producing a noisy, undifferentiated spatial map rather than distinct compartments. For practical use, inspect the smaller modules (5–13 genes) where high module scores should be restricted to specific anatomical zones: a stromal module scoring high in LTBP2/PDGFRB-rich fibroblast belts, or an immune module peaking in the lymphocyte-dense zones around tumor margins.

### 5c. Subcellular spatial domains

SOM can also be applied at the sub-cellular level. Each transcript's position is normalized relative to its cell's centroid and equivalent radius, making cells of different sizes comparable. The SOM then discovers recurring *spatial patterns* across all cells: a node at the grid center may correspond to nuclear accumulation, while peripheral nodes may represent membrane-enriched transcripts.

Results are stored as:
- `obs['som_subcellular_dominant']` — the dominant domain per cell (useful for tissue-level spatial maps)
- `obsm['som_subcellular_fractions']` — full `(n_cells, n_domains)` domain composition matrix


In [ ]:
# Load the full transcript table into mdata['xenium'].uns['transcripts']
# read_xenium_transcripts returns None when inplace=True (default); transcripts
# are stored in uns, not returned.
ab.io.read_xenium_transcripts(mdata)
transcripts = mdata['xenium'].uns['transcripts']

# 5x5 grid = 25 subcellular domains; train on normalized intracellular (dx/r, dy/r) positions
tx_domains = ab.tl.som_subcellular_domains(mdata, transcripts, x=5, y=5, seed=42)
print(f'Assigned {len(tx_domains)} transcripts to {tx_domains["domain"].nunique()} domains')
xen.obs['som_subcellular_dominant'].value_counts().head(8)

9,131,749 of 10,606,835 transcripts (86%) passed quality filters and were assigned to the 5×5 = 25 subcellular domains. The dominant domain for each cell is a corner or edge node (`0_4`, `1_2`, `3_1`, `4_1`) rather than the center, reflecting that most transcripts in FFPE Xenium data localize to the cell periphery and cytoplasm rather than the nucleus center. The 5×5 grid (25 domains) is intentionally small: with only 2 input dimensions (normalized x and y), a finer grid would over-partition the smooth intracellular position distribution. Domain `0_4` dominates 13,958 cells (9%), likely the top-right corner of the intracellular coordinate frame, corresponding to one of the four peripheral quadrants.

In [ ]:
# Draw transcript positions inside cell outlines, coloured by domain label
# Reveals whether domains correspond to nuclear / cytoplasmic / peripheral regions
ab.pl.som_subcellular_plot(mdata, tx_domains, max_cells=150).show()

In [ ]:
# Tissue-level view: which subcellular pattern dominates each cell?
# Cells of the same type or in the same neighbourhood may share a dominant domain
(ab.pl.points(mdata, color='som_subcellular_dominant')
 | ab.pl.boundaries(mdata, color='som_subcellular_dominant')).show()

The tissue-level view of dominant subcellular domain reveals whether intracellular transcript organization is spatially structured or cell-autonomous. If cells in the same tissue compartment share a dominant domain (e.g., all tumor cells in a nest pointing their dominant transcript accumulation in the same direction), it suggests a collective polarity signal — perhaps aligned with a gradient or a shared vasculature axis. Random domain assignment across neighbors indicates transcript localization is cell-intrinsic rather than tissue-level coordinated. The boundary overlay separates the cell polygon from its domain assignment, making it easier to see if large epithelial cells vs. small immune cells differ in their dominant intracellular pattern.

## Summary

Two findings, both pointing beyond the discrete-cluster summary from `01_core_workflow.ipynb`: the PAGA/DPT trajectory shows cluster identity is at least partly a continuum; the spatial gene module identifies a co-expression structure independent of cluster labels. Either finding could seed a hypothesis about cell-type-driven contact effects explored in `03_neighborhood_analysis.ipynb`.